In [1]:
# === 自动探测项目根目录（兼容任意启动方式） ===
from pathlib import Path

def find_project_root():
    if '__vsc_ipynb_file__' in globals():
        start = Path(globals()['__vsc_ipynb_file__']).parent
    else:
        start = Path.cwd()

    # 向上查找包含 data 文件夹的目录
    for path in [start] + list(start.parents):
        if (path / "data").exists():
            return path
    raise FileNotFoundError("❌ 找不到项目根目录！请确保项目包含 data 文件夹。")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
print("项目根目录已定位")

项目根目录已定位


In [2]:
import pandas as pd
# 定义列名（原始数据没有表头）
columns = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']

# 读取数据
df = pd.read_csv(DATA_DIR / 'UserBehavior_sample.csv', names=columns)

# 1. 查看数据基本信息
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 5 columns):
 #   Column         Non-Null Count    Dtype
---  ------         --------------    -----
 0   user_id        1000000 non-null  int64
 1   item_id        1000000 non-null  int64
 2   category_id    1000000 non-null  int64
 3   behavior_type  1000000 non-null  str  
 4   timestamp      1000000 non-null  int64
dtypes: int64(4), str(1)
memory usage: 38.1 MB


In [3]:
# 2. 查看示例数据
df.head()

,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [4]:
# 3. 查看数值型字段分布
df.describe()

,user_id,item_id,category_id,timestamp
count,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06
mean,4.299565e+05,2.577589e+06,2.706038e+06,1.511962e+09
std,4.353203e+05,1.488887e+06,1.464067e+06,2.305482e+05
min,1.000000e+00,7.200000e+01,2.171000e+03,1.505118e+09
25%,1.091610e+05,1.292238e+06,1.343555e+06,1.511762e+09
50%,1.216220e+05,2.575682e+06,2.693696e+06,1.511965e+09
75%,1.005419e+06,3.861783e+06,4.145813e+06,1.512180e+09
max,1.018011e+06,5.163057e+06,5.161669e+06,1.512317e+09


In [5]:
# 4. 检查时间跨度
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s') + pd.Timedelta(hours=8)
print('最早:', df['datetime'].min())
print('最晚:', df['datetime'].max())

最早: 2017-09-11 16:16:39
最晚: 2017-12-04 00:00:06


**注意：** 在这里，我们发现，打印出来的最早和最晚的时间范围，并不符合阿里云天池官方文档中说明的 **2017年11月25号到2017年12月3号**的数据范围，所以我们在后面的 **02_data_cleaning.ipynb** 里会重新定义时间范围 （北京时间UTC+8), 并**将不属于这个时间范围内的数据清洗掉**。

In [6]:
# 5. 记录基础统计数字

print(f"总行数: {len(df):,}")
print(f"总用户数: {df['user_id'].nunique():,}")
print(f"总商品数: {df['item_id'].nunique():,}")
print(f"总品类数: {df['category_id'].nunique():,}")
print(f"行为类型: {df['behavior_type'].unique()}")

总行数: 1,000,000
总用户数: 9,739
总商品数: 399,114
总品类数: 5,796
行为类型: <StringArray>
['pv', 'fav', 'buy', 'cart']
Length: 4, dtype: str


In [7]:
# 检查是否有重复行
duplicate_rows = df.duplicated().sum()
print(f"重复行数: {duplicate_rows}")

# 检查各列是否有空值
print(f"各列空值:\n{df.isnull().sum()}")

# 检查各列是否只有一个值（单值列）
for col in df.columns:
    print(f"{col}: {df[col].nunique()} 个唯一值")

# 检查是否有重复列名（这组数据没有表头，所以不会重复）
print(f"列名重复: {df.columns.duplicated().sum()}")

重复行数: 0
各列空值:
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
datetime         0
dtype: int64
user_id: 9739 个唯一值
item_id: 399114 个唯一值
category_id: 5796 个唯一值
behavior_type: 4 个唯一值
timestamp: 499049 个唯一值
datetime: 499049 个唯一值
列名重复: 0


In [8]:
# 将 Unix 时间戳转换为北京时间（UTC+8）
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s') + pd.Timedelta(hours=8)

# 定义分析时间范围（按官方文档说明）
start_date = '2017-11-25'
end_date = '2017-12-04'  # 包含 12-03 全天

# 过滤数据
df = df[(df['datetime'] >= start_date) & (df['datetime'] < end_date)]

# 确认过滤结果
print(f"过滤后总行数: {len(df):,}")
print(f"时间范围: {df['datetime'].min()} 至 {df['datetime'].max()}")

过滤后总行数: 999,530
时间范围: 2017-11-25 00:00:00 至 2017-12-03 23:59:50
